# N5 — Integration Notebook
**Competency** C16 · **Artifact** Final Integration Report · **Input** all exports

Assemble the validated system report end to end: reproduce each stage's headline metric, verify all thresholds, diagnose any residual.


## 1-2. Purpose & inputs
Inputs: the B-series CSVs (kinematics, hydraulics, control, validation). This notebook is the capstone roll-up.


In [1]:
from pathlib import Path
import csv, json, math, urllib.request
# Reference CSVs live in docs/figures. In-repo this resolves directly; on Colab the
# files are pulled from raw GitHub. Students can instead drop their own demo exports in DATA.
REPO_RAW='https://raw.githubusercontent.com/alibulentkoc/parallel-kinematics-hydraulics/main/docs/figures'
DATA = Path('../figures') if Path('../figures').exists() else Path('figures')
DATA.mkdir(exist_ok=True) if DATA==Path('figures') else None
def _ensure(name):
    p=DATA/name
    if not p.exists():
        try: urllib.request.urlretrieve(f'{REPO_RAW}/{name}', p); print('fetched', name)
        except Exception as e: raise FileNotFoundError(f'{name} not found locally and fetch failed: {e}')
    return p
def load_csv(name):
    rows=[]
    with open(_ensure(name)) as f:
        reader=csv.reader(l for l in f if not l.startswith('#'))
        header=next(reader)
        for r in reader:
            if r: rows.append(dict(zip(header,r)))
    return header, rows
def col(rows,k,cast=float):
    return [cast(r[k]) for r in rows if r.get(k) not in (None,'')]


In [2]:
B=0.6
def ik2(x,y): return [math.hypot(x+B,y), math.hypot(x-B,y)]
def fk2(L1,L2):
    x=(L1*L1-L2*L2)/(4*B); y=math.sqrt(max(0.0,L1*L1-(x+B)**2)); return x,y
def rmse(a,b): return math.sqrt(sum((p-q)**2 for p,q in zip(a,b))/len(a))


## 3-4. Reproduce each stage's headline metric


In [3]:
summary={}
# kinematics round-trip
L=ik2(0.10,0.70); x2,y2=fk2(*L); summary['kin_roundtrip_m']=math.hypot(x2-0.10,y2-0.70)
# hydraulics phi
_,b4=load_csv('B4-force-area.csv'); summary['phi']=float([r for r in b4 if r['rod_mm']=='22'][0]['phi'])
# control tracking + settling
_,b9=load_csv('B9-coordinated-tracking.csv')
summary['track_rmse_mm']=math.sqrt(sum((float(r['target_x'])-float(r['actual_x']))**2+(float(r['target_y'])-float(r['actual_y']))**2 for r in b9)/len(b9))*1000
_,b10=load_csv('B10-tuned-response.csv'); tcol=col(b10,'t'); xcol=col(b10,'x'); tol=0.10*0.02; st=0.0
for ti,xi in zip(tcol,xcol):
    if abs(0.10-xi)>tol: st=ti
summary['settling_s']=st
# validation
_,b11=load_csv('B11-twin-vs-rig.csv'); tw=col(b11,'twin_x'); rg=col(b11,'rig_x'); twP=col(b11,'twin_P'); rgP=col(b11,'rig_P')
summary['pos_rmse_mm']=rmse(tw,rg)*1000
summary['pressure_err_pct']=100*rmse(twP,rgP)/(sum(rgP)/len(rgP))
print(json.dumps({k:round(v,4) for k,v in summary.items()}, indent=1))


{
 "kin_roundtrip_m": 0.0,
 "phi": 1.434,
 "track_rmse_mm": 8.3169,
 "settling_s": 0.471,
 "pos_rmse_mm": 0.0,
 "pressure_err_pct": 0.0
}


## 5-6. Verify the full acceptance bundle


In [4]:
gates={
  'kin round-trip < 1e-6 m': summary['kin_roundtrip_m']<1e-6,
  'phi <= 1.6': summary['phi']<=1.6,
  'tracking RMSE <= 10 mm': summary['track_rmse_mm']<=10.0,
  'settling <= 2.5 s': summary['settling_s']<=2.5,
  'pos RMSE <= 10 mm': summary['pos_rmse_mm']<=10.0,
  'pressure err <= 15%': summary['pressure_err_pct']<=15.0,
}
for k,v in gates.items(): print(f'  [{"PASS" if v else "FAIL"}] {k}')
assert all(gates.values()), 'final integration gates'
print('\nALL GATES PASS — system validated')


  [PASS] kin round-trip < 1e-6 m
  [PASS] phi <= 1.6
  [PASS] tracking RMSE <= 10 mm
  [PASS] settling <= 2.5 s
  [PASS] pos RMSE <= 10 mm
  [PASS] pressure err <= 15%

ALL GATES PASS — system validated


## 7. Export Final Report package


In [5]:
with open('final-report-package.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['stage','metric','value','pass'])
    w.writerow(['kinematics','roundtrip_m',summary['kin_roundtrip_m'],summary['kin_roundtrip_m']<1e-6])
    w.writerow(['hydraulics','phi',round(summary['phi'],3),summary['phi']<=1.6])
    w.writerow(['control','track_rmse_mm',round(summary['track_rmse_mm'],2),summary['track_rmse_mm']<=10])
    w.writerow(['control','settling_s',round(summary['settling_s'],2),summary['settling_s']<=2.5])
    w.writerow(['validation','pos_rmse_mm',round(summary['pos_rmse_mm'],2),summary['pos_rmse_mm']<=10])
    w.writerow(['validation','pressure_err_pct',round(summary['pressure_err_pct'],2),summary['pressure_err_pct']<=15])
print('exported final-report-package.csv')


exported final-report-package.csv
